# Healthcare Data Governance and Cleaning

This notebook performs reproducible data-quality validation on the synthetic/de-identified patient dataset.

**Goal:** Validate structure, data types, missing values, duplicate PatientIDs, ranges, categories, blood-pressure format, cholesterol, and medication dosage without inventing clinical values.

In [ ]:
import pandas as pd
import re

df = pd.read_csv("healthcare_patients_cleaned.csv")

df

## 1. Basic Structure and Data Types

This section checks the number of records, number of fields, column names, and data types.

In [ ]:
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nColumn names:")
print(list(df.columns))

print("\nData types:")
print(df.dtypes)

## 2. Missing-Value Check

This section checks every field for missing values.

In [ ]:
missing = df.isnull().sum()

print("Missing values by field:")
print(missing)

print("\nTotal missing cells:", int(missing.sum()))

## 3. Duplicate PatientID Check

PatientID values should be unique so that records are not accidentally duplicated.

In [ ]:
duplicate_count = int(df["PatientID"].duplicated().sum())

print("Duplicate PatientIDs:", duplicate_count)

## 4. PatientID Format Validation

The supplied synthetic PatientIDs are expected to follow the format `PAT-###`, such as `PAT-201`.

In [ ]:
patient_id_valid = df["PatientID"].astype(str).str.match(r"^PAT-\d{3}$")

invalid_patient_ids = int((~patient_id_valid).sum())

print("Invalid PatientID formats:", invalid_patient_ids)

## 5. Range and Category Validation

The following checks validate age, treatment duration, readmission status, cholesterol, and medication dosage.

These checks identify structurally invalid values. They do not attempt to make unsupported clinical corrections.

In [ ]:
invalid_age = int(((df["Age"] < 0) | (df["Age"] > 120)).sum())

invalid_duration = int((df["TreatmentDurationDays"] <= 0).sum())

invalid_readmitted = int((~df["Readmitted"].isin(["Yes", "No"])).sum())

invalid_cholesterol = int((df["CholesterolLevel"] < 0).sum())

invalid_dosage = int((df["DosageMg"] < 0).sum())

print("Invalid ages:", invalid_age)
print("Invalid treatment durations:", invalid_duration)
print("Invalid Readmitted values:", invalid_readmitted)
print("Invalid cholesterol values:", invalid_cholesterol)
print("Invalid dosage values:", invalid_dosage)

## 6. Blood Pressure Format Validation

Blood pressure values are expected to contain two numeric components separated by `/`, for example `138/88`.

In [ ]:
bp_valid = df["BloodPressure"].astype(str).str.match(r"^\d{2,3}/\d{2,3}$")

invalid_bp = int((~bp_valid).sum())

print("Invalid blood pressure formats:", invalid_bp)

## 7. Final Validation Summary

The following summary combines all validation checks performed on the dataset.

In [ ]:
summary = {
    "Total records": len(df),
    "Duplicate PatientIDs": duplicate_count,
    "Missing cells": int(missing.sum()),
    "Invalid PatientID formats": invalid_patient_ids,
    "Invalid ages": invalid_age,
    "Invalid treatment durations": invalid_duration,
    "Invalid Readmitted values": invalid_readmitted,
    "Invalid blood pressure formats": invalid_bp,
    "Invalid cholesterol values": invalid_cholesterol,
    "Invalid dosage values": invalid_dosage
}

validation_summary = pd.Series(summary)

validation_summary

## Validation Conclusion

The supplied dataset contains 10 records and passed the defined structural and data-quality validation checks.

No missing values, duplicate PatientIDs, invalid PatientID formats, invalid ages, invalid treatment durations, invalid Readmitted categories, invalid blood pressure formats, invalid cholesterol values, or invalid dosage values were identified.

Because no invalid records required correction, no clinical values were deleted, replaced, or imputed.

The Levothyroxine dosage of `0.05 mg` was retained as supplied. Medication dosage interpretation is medication-specific, and no conversion was performed because the task does not provide an explicit clinical conversion rule.

The validated dataset is therefore used as the cleaned dataset for this exercise.